In [ ]:
import numpy as np
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

def segmentVolley(Filename):
    # Backhand data set
    bData = np.genfromtxt(Filename, delimiter=',', skip_header=1)
    bPacketNumber = bData[:, 0]
    bTimedomain = bPacketNumber / 128  # because the frequency used is 128Hz
    bGyroX = bData[:, 1]
    bGyroY = bData[:, 2]
    bGyroZ = bData[:, 3]

    # Absolute magnitude of gyro data and curve smoothing
    bGyroMean = np.sqrt((bGyroX**2) + (bGyroY**2) + (bGyroZ**2))  # absolute magnitude backhand
    windowSize = 40
    bGyroMeanMovav = uniform_filter1d(bGyroMean, size=windowSize)  # moving average backhand
    bpks, _ = find_peaks(bGyroMeanMovav, prominence=200, distance=252, height=300)  # peak distance of 2s and min peak height of 300

    # Saving each shot into separate column of one big array
    # shot length is assumed to be 5s in length
    SignalTime = np.zeros((433, len(bpks)))
    SignalX = np.zeros((433, len(bpks)))
    SignalY = np.zeros((433, len(bpks)))
    SignalZ = np.zeros((433, len(bpks)))

    for j in range(len(bpks)):
        timeIndex = bpks[j]  # select first time value of peak
        startSignal = timeIndex - 216  # subtract 2.3s worth of data points from peak to obtain time when shot begins
        endSignal = timeIndex + 216  # add 2.5s worth of data points from peak to obtain end of signal
        if j == 0:
            startSignal = 0
            endSignal = 433
        elif j == len(bpks) - 1:
            startSignal = len(bTimedomain) - 432
            endSignal = len(bTimedomain)

        SignalTime[:, j] = bTimedomain[startSignal:endSignal]  # 1 shot in time index to be saved in Signal matrix
        SignalX[:, j] = bGyroX[startSignal:endSignal]
        SignalY[:, j] = bGyroY[startSignal:endSignal]
        SignalZ[:, j] = bGyroZ[startSignal:endSignal]

    # Plotting
    plt.figure()

            # Plotting
        # Plotting all axis in time domain
    for plotIdx in range(SignalX.shape[1]):
        plt.plot(SignalX[:, plotIdx], 'r')  # overlay all forehand shots onto 1 curve
        plt.plot(SignalY[:, plotIdx], 'g')  # overlay all forehand shots onto 1 curve
        plt.plot(SignalZ[:, plotIdx], 'b')  # overlay all forehand shots onto 1 curve
        plt.legend(['\omega_x', '\omega_y', '\omega_z'])
    plt.xlabel('Time (s)')  # x-axis label
    plt.xticks([126, 252, 378, 504], ['1', '2', '3', '4'])
    plt.ylabel('Gyroscope (°/s)')  # y-axis label
    plt.legend().set_visible(False)

    # Comment the following lines if youdont  want to plot the data by axis
    plt.figure()
    for plotIdx in range(SignalX.shape[1]):
        plt.plot(SignalX[:, plotIdx], 'r')  # overlay all forehand shots onto 1 curve
        plt.xlabel('Time (s)')  # x-axis label
        plt.ylabel('Gyroscope (°/s)')  # y-axis label
        plt.title('X gyroscope in Time Domain')
    plt.figure()
    for plotIdx in range(SignalX.shape[1]):
        plt.plot(SignalY[:, plotIdx], 'g')  # overlay all forehand shots onto 1 curve
        plt.xlabel('Time (s)')  # x-axis label
        plt.ylabel('Gyroscope (°/s)')  # y-axis label
        plt.title('Y Gyroscope (°/s) in Time Domain')
    plt.figure()
    for plotIdx in range(SignalX.shape[1]):
        plt.plot(SignalZ[:, plotIdx], 'b')  # overlay all forehand shots onto 1 curve
        plt.xlabel('Time (s)')  # x-axis label
        plt.ylabel('Gyroscope (°/s)')  # y-axis label
        plt.title('Z Gyroscope (°/s) in Time Domain')

    return SignalTime, SignalX, SignalY, SignalZ